## Importing Libraries

In [6]:
import pandas as pd
import numpy as np
from pathlib import Path
import clickhouse_connect

### Define IMEI Date Filter

In [7]:
IMEI_DATE_FILTER = """
    WHERE imei_first_seen >= '2026-06-01'
      AND imei_first_seen < '2026-07-01'
"""

## Importing KYC data

In [6]:
# 1. Define your base directory using Pathlib (makes it easy to update later)
BASE_DIR = Path("/Volumes/E$/KYC/Merged Clean Dumps/2026")

# 2. Load only two DataFrames to save massive amounts of RAM  ---- Change Dataframe Names ‼️‼️‼️‼️‼️‼️‼️‼️‼️‼️‼️‼️‼️‼️
df = pd.read_parquet(BASE_DIR / "df_05_26.parquet")
df_NID = pd.read_parquet(BASE_DIR / "df_NID_05_26.parquet")

In [7]:
df.head()

In [8]:
df_NID.head()

In [9]:
# Keep only rows where the MSISDN is exactly 10 characters long
df_NID = df_NID[df_NID['msisdn'].astype(str).str.len() == 10]
df = df[df['msisdn'].astype(str).str.len() == 10]

# Ensure the column is treated as text, then replace the leading '0' with '256'
df_NID['msisdn'] = df_NID['msisdn'].astype(str).str.replace(r'^0', '256', regex=True)
df['msisdn'] = df['msisdn'].astype(str).str.replace(r'^0', '256', regex=True)


# Drop unecessary columns to save memory
df = df.drop(columns=['surname', 'first_name'])
df_NID = df_NID.drop(columns=['surname', 'first_name'])

## Importing GSMA data

In [8]:
# Connect to ClickHouse with resource limits
client = clickhouse_connect.get_client(
    host='192.168.1.95',
    port=8123,
    username='default',
    password='',
    database='ceir',
    settings={
        'max_memory_usage': 4000000000,  # 4GB max
        'max_threads': 2,
        'priority': 5
    }
)

# Step 1: Fetch data from the gsma_devices table
gsma_query = """
SELECT
    tac,
    oem,
    brand,
    model,
    marketing_name,
    device_type,
    os_family,
    os_version,
    sim_slots,
    has_2g,
    has_3g,
    has_4g,
    has_5g,
    year_released
FROM gsma_devices
"""
gsma_df = client.query_df(gsma_query)

In [9]:
len(gsma_df)

290402

In [10]:
gsma_df.head()

,tac,oem,brand,model,marketing_name,device_type,os_family,os_version,sim_slots,has_2g,has_3g,has_4g,has_5g,year_released
0,00100100,Not Known,Not Known,G410,,Handheld,,,0,0,0,0,0,0
1,00100200,Not Known,Not Known,A53,,Handheld,,,0,0,0,0,0,2004
2,00100300,Not Known,Not Known,TBD (AAB-1880030-BV),,Handheld,,,0,0,0,0,0,0
3,00100400,Not Known,Not Known,RM-669,,Handheld,Nokia OS,,0,1,0,0,0,2009
4,00100500,Not Known,Not Known,M930 NA DB,,Handheld,,,0,0,0,0,0,0


## Importing Fake IMEIs Table

In [13]:
# Connect to ClickHouse with resource limits
client = clickhouse_connect.get_client(
    host='192.168.1.95',
    port=8123,
    username='default',
    password='',
    database='ceir_gold',
    settings={
        'max_memory_usage': 2000000000,  # 2GB max
        'max_threads': 2,
        'priority': 5
    }
)

# Step 1: Fetch data from the imeis_fake table
fake_query = f"SELECT * FROM imeis_fake_v {IMEI_DATE_FILTER}"

fake_df = client.query_df(fake_query)

In [14]:
len(fake_df)

In [15]:
fake_df.head(100)

In [16]:
# Drop Empty Device Type Column
fake_df = fake_df.drop(columns=['device_type'])


# Re Arrange Columns
fake_df=fake_df[['imei_first_seen', 'last_seen', 'imei', 'imei_status', 'imsi', 'msisdn', 'rat', 'cgi']]

In [17]:
# Step 1: attach the rich NID KYC (gender, age, district) where msisdn is NID-registered
fake_df = fake_df.merge(df_NID, on='msisdn', how='left')

# Step 2: bring in df for the fallback, using suffixes to keep the collisions separate
fake_df = fake_df.merge(df, on='msisdn', how='left', suffixes=('', '_df'))

# Step 3: coalesce — df_NID value wins, df fills only where df_NID was NaN
overlap = ['id_type', 'id_number', 'prefix', 'mno']  # columns that exist in both df_NID and df
for col in overlap:
    target = fake_df[col].astype('object')        # drop categorical restriction
    source = fake_df[f'{col}_df'].astype('object')
    fake_df[col] = target.fillna(source)

# Step 4: drop the now-redundant _df columns
fake_df = fake_df.drop(columns=[f'{col}_df' for col in overlap])

# Step 5: convert the relevant columns back to categorical for memory efficiency
for col in ['id_type', 'prefix', 'mno']:
    fake_df[col] = fake_df[col].astype('category')

In [18]:
# generate mno from imsi where missing, using the standard prefix mapping
# ======================================================================
# Only touch rows where mno is currently missing
mask = fake_df['mno'].isna()

# Make sure imsi is string so .str works reliably
imsi = fake_df['imsi'].astype('string')

# Derive operator from the IMSI prefix
mtn    = mask & imsi.str.startswith('64110')
hamilton    = mask & imsi.str.startswith('64120')
talkio = mask & imsi.str.startswith('64108')
airtel = mask & (imsi.str.startswith('64101') | imsi.str.startswith('64122'))

# If mno is categorical, add the new categories before assigning (same trap as before)
if isinstance(fake_df['mno'].dtype, pd.CategoricalDtype):
    fake_df['mno'] = fake_df['mno'].cat.add_categories(
        [c for c in ['MTN', 'AIRTEL', 'HAMILTON', 'TALKIO'] if c not in fake_df['mno'].cat.categories]
    )

fake_df.loc[mtn, 'mno']    = 'MTN'
fake_df.loc[hamilton, 'mno'] = 'HAMILTON'
fake_df.loc[talkio, 'mno'] = 'TALKIO'
fake_df.loc[airtel, 'mno'] = 'AIRTEL'

In [19]:
# Enrich Dataset with MCC - to Have Country Information for each IMSI

# 1) Load MCC lookup table
mcc_lu = pd.read_csv("/Users/wmuheki/Documents/Projects/Analytics/ceir/clean_dumps/MCC_Each_country.csv", dtype=str)
mcc_lu.head()


# 2) Normalize lookup columns
mcc_lu = mcc_lu.rename(columns={"MCC": "mcc", "Country": "country"})
mcc_lu["mcc"] = mcc_lu["mcc"].astype("string").str.strip()
mcc_lu["country"] = mcc_lu["country"].astype("string").str.strip()
mcc_lu = mcc_lu.drop_duplicates(subset=["mcc"])


# 3) Extract MCC from IMSI (first 3 digits)
fake_df["imsi"] = fake_df["imsi"].astype("string")

fake_df["mcc"] = (
    fake_df["imsi"]
    .str.replace(r"\D+", "", regex=True)  # keep digits only
    .str.slice(0, 3)
)


# 4) Merge Country Code Data Frame with the Roaming Data sets
fake_df = fake_df.merge(
    mcc_lu[["mcc", "country"]],
    how="left",
    on="mcc"
)


# 5) Optional: fill unknowns
fake_df["country"] = fake_df["country"].fillna("UNKNOWN")


In [20]:
fake_df.head()

## Importing Genuine IMEIs Table

### Change to Eventually Pick only one month !!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!

In [21]:
# Connect to ClickHouse with resource limits
client = clickhouse_connect.get_client(
    host='192.168.1.95',
    port=8123,
    username='default',
    password='',
    database='ceir_gold',
    settings={
        'max_memory_usage': 4000000000,  # 4GB max
        'max_threads': 2,
        'priority': 5
    }
)

# Step 1: Fetch data from the imeis_genuine table
genuine_query = f"SELECT * FROM imeis_genuine_v {IMEI_DATE_FILTER}"

genuine_df = client.query_df(genuine_query)

In [ ]:
len(genuine_df)

In [ ]:
# Drop Device Type Column - we shall regenerate it from GSMA TAC data
genuine_df = genuine_df.drop(columns=['device_type'])

# Convert IMEI to string, extract the first 8 characters, and create the 'tac' column
genuine_df['tac'] = genuine_df['imei'].astype(str).str[:8]

# Re Arrange Columns
genuine_df=genuine_df[['imei_first_seen','last_seen', 'tac', 'imei', 'imei_status', 'imsi', 'msisdn', 'rat', 'cgi']]

# Merge with GSMA Data to get device details for fake IMEIs
genuine_df = genuine_df.merge(gsma_df, on='tac', how='left')

In [ ]:
# Step 1: attach the rich NID KYC (gender, age, district) where msisdn is NID-registered
genuine_df = genuine_df.merge(df_NID, on='msisdn', how='left')

# Step 2: bring in df for the fallback, using suffixes to keep the collisions separate
genuine_df = genuine_df.merge(df, on='msisdn', how='left', suffixes=('', '_df'))

# Step 3: coalesce — df_NID value wins, df fills only where df_NID was NaN
overlap = ['id_type', 'id_number', 'prefix', 'mno']  # columns that exist in both df_NID and df
for col in overlap:
    target = genuine_df[col].astype('object')        # drop categorical restriction
    source = genuine_df[f'{col}_df'].astype('object')
    genuine_df[col] = target.fillna(source)

# Step 4: drop the now-redundant _df columns
genuine_df = genuine_df.drop(columns=[f'{col}_df' for col in overlap])

# Step 5: convert the relevant columns back to categorical for memory efficiency
for col in ['id_type', 'prefix', 'mno']:
    genuine_df[col] = genuine_df[col].astype('category')

In [ ]:
# generate mno from imsi where missing, using the standard prefix mapping
# ======================================================================
# Only touch rows where mno is currently missing
mask = genuine_df['mno'].isna()

# Make sure imsi is string so .str works reliably
imsi = genuine_df['imsi'].astype('string')

# Derive operator from the IMSI prefix
mtn    = mask & imsi.str.startswith('64110')
hamilton    = mask & imsi.str.startswith('64120')
talkio = mask & imsi.str.startswith('64108')
airtel = mask & (imsi.str.startswith('64101') | imsi.str.startswith('64122'))

# If mno is categorical, add the new categories before assigning (same trap as before)
if isinstance(genuine_df['mno'].dtype, pd.CategoricalDtype):
    genuine_df['mno'] = genuine_df['mno'].cat.add_categories(
        [c for c in ['MTN', 'AIRTEL', 'HAMILTON', 'TALKIO'] if c not in genuine_df['mno'].cat.categories]
    )

genuine_df.loc[mtn, 'mno']    = 'MTN'
genuine_df.loc[hamilton, 'mno'] = 'HAMILTON'
genuine_df.loc[talkio, 'mno'] = 'TALKIO'
genuine_df.loc[airtel, 'mno'] = 'AIRTEL'

In [ ]:
# Enrich Dataset with MCC - to Have Country Information for each IMSI

# 1) Load MCC lookup table
mcc_lu = pd.read_csv("/Users/wmuheki/Documents/Projects/Analytics/ceir/clean_dumps/MCC_Each_country.csv", dtype=str)
mcc_lu.head()


# 2) Normalize lookup columns
mcc_lu = mcc_lu.rename(columns={"MCC": "mcc", "Country": "country"})
mcc_lu["mcc"] = mcc_lu["mcc"].astype("string").str.strip()
mcc_lu["country"] = mcc_lu["country"].astype("string").str.strip()
mcc_lu = mcc_lu.drop_duplicates(subset=["mcc"])


# 3) Extract MCC from IMSI (first 3 digits)
genuine_df["imsi"] = genuine_df["imsi"].astype("string")

genuine_df["mcc"] = (
    genuine_df["imsi"]
    .str.replace(r"\D+", "", regex=True)  # keep digits only
    .str.slice(0, 3)
)


# 4) Merge Country Code Data Frame with the Roaming Data sets
genuine_df = genuine_df.merge(
    mcc_lu[["mcc", "country"]],
    how="left",
    on="mcc"
)

# 5) Optional: fill unknowns
genuine_df["country"] = genuine_df["country"].fillna("UNKNOWN")


In [ ]:
#Change Data type to remove decimal points and convert to integers
genuine_df['sim_slots'] = genuine_df['sim_slots'].astype('Int64')
genuine_df['has_2g'] = genuine_df['has_2g'].astype('Int64')
genuine_df['has_3g'] = genuine_df['has_3g'].astype('Int64')
genuine_df['has_4g'] = genuine_df['has_4g'].astype('Int64')
genuine_df['has_5g'] = genuine_df['has_5g'].astype('Int64')
genuine_df['year_released'] = genuine_df['year_released'].astype('Int64')

In [ ]:
genuine_df.head()

## Cloned IMEIs

In [25]:
# Connect to ClickHouse with resource limits
client = clickhouse_connect.get_client(
    host='192.168.1.95',
    port=8123,
    username='default',
    password='',
    database='ceir_gold',
    settings={
        'max_memory_usage': 4000000000,  # 4GB max
        'max_threads': 2,
        'priority': 5
    }
)

# Step 1: Fetch data from the imeis_genuine table
clone_query = "SELECT * FROM cloned_imeis_v"

clone_df = client.query_df(clone_query)

In [26]:
len(clone_df)

37634

In [27]:
# Drop Device Type Column - we shall regenerate it from GSMA TAC data
clone_df = clone_df.drop(columns=['device_type'])

# Drop IMSIs and MSISDNs columns - we do not have statitics use cases - we can always retain them incase needed in the future
clone_df = clone_df.drop(columns=['imsis', 'msisdns'])

# Rename first_detected_at and last_change_at to imei_first_seen and last_seen for consistency with other datasets
clone_df = clone_df.rename(columns={'first_detected_at': 'imei_first_seen', 'last_change_at': 'last_seen'})

# Convert IMEI to string, extract the first 8 characters, and create the 'tac' column
clone_df['tac'] = clone_df['imei'].astype(str).str[:8]

# Re Arrange Columns
clone_df=clone_df[['imei_first_seen','last_seen', 'tac', 'imei', 'imsi_count', 'msisdn_count']]

# Merge with GSMA Data to get device details for fake IMEI
clone_df = clone_df.merge(gsma_df, on='tac', how='left')

In [28]:
clone_df.head(5)

,imei_first_seen,last_seen,tac,imei,imsi_count,msisdn_count,oem,brand,model,marketing_name,device_type,os_family,os_version,sim_slots,has_2g,has_3g,has_4g,has_5g,year_released
0,2026-07-01 21:02:08,2026-07-01 21:43:52,35804817,35804817426274,2,2,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,NaN
1,2026-07-01 17:11:48,2026-07-01 21:20:53,35918236,35918236533982,2,2,Samsung Korea,Samsung,SM-A065F/DS,Galaxy A06,Smartphone,Android,14,2.0,1.0,1.0,1.0,0.0,2024.0
2,2026-07-02 10:33:23,2026-07-02 10:34:22,86268903,86268903193491,2,2,Not Known,vivo,vivo X7Plus,X7 Plus,Smartphone,Android,5.1,0.0,1.0,1.0,1.0,0.0,2016.0
3,2026-07-01 21:17:09,2026-07-01 22:39:06,35074853,35074853915984,2,2,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,NaN
4,2026-07-01 18:22:47,2026-07-02 06:26:13,35627435,35627435026873,3,3,Itel Technology Limited,itel,it5606,it5606,Mobile Phone/Feature phone,,,2.0,1.0,0.0,0.0,0.0,2018.0


In [ ]:

# Export the enriched datasets to Parquet for future use

fake_df.to_parquet("/Volumes/E$/CEIR/Clean Dumps/Fake/fake_2026-06.parquet") # Update File Name as Needed
genuine_df.to_parquet("/Volumes/E$/CEIR/Clean Dumps/Genuine/genuine_2026-06.parquet") # Update File Name as Needed



clone_df.to_parquet("/Volumes/E$/CEIR/Clean Dumps/Cloned/all_cloned.parquet") # No need to update file name as this is a cumulative dataset of all cloned IMEIs
